In [1]:
from pathlib import Path
_template = str(Path(__vsc_ipynb_file__).parent.parent / 'template.ipynb')
%run "$_template"

In [2]:
df_train = pd.read_csv("../data/modeling/train.csv")
df_test = pd.read_csv("../data/modeling/test.csv")

display(df_train.head())
print("train shape:", df_train.shape)
print("test shape:", df_test.shape)

,Ox_Chamber,type,Temp_OXid,ppm,Pressure,Oxid_time,thickness,photo_soft_Chamber,resist_target,N2_HMDS,...,RTA_Temp,Etching_Chamber,Thin F4,Thin F3,Thin F2,Thin F1,Temp_Etching,Source_Power,Selectivity,is_low_yield
0,2,wet,1031.14,34.06,0.58,129,709.38,3,1.771,14.941,...,153,2,389.0,1522.0,3628.0,5738.0,71.780,51.317,1.140,0
1,2,wet,1044.90,29.87,0.51,165,727.94,1,0.788,13.768,...,154,3,316.0,1564.0,3646.0,5735.0,71.232,50.263,1.042,0
2,1,wet,1138.86,32.79,0.43,62,699.46,1,1.783,13.909,...,154,3,251.0,1606.0,3676.0,5756.0,70.774,50.737,1.198,0
3,1,wet,1019.22,35.03,0.47,210,719.17,3,2.114,16.638,...,154,3,278.0,1526.0,3654.0,5713.0,71.851,51.553,1.069,0
4,3,wet,1133.71,26.68,0.32,106,701.73,2,1.531,16.621,...,155,3,317.0,1449.0,3617.0,5739.0,70.515,49.539,1.053,0


train shape: (10747, 44)
test shape: (4607, 44)


In [3]:
photo_softbake = [
    'photo_soft_Chamber',
    'resist_target',
    'N2_HMDS',
    'pressure_HMDS',
    'temp_HMDS',
    'temp_HMDS_bake',
    'time_HMDS_bake',
    'spin1',
    'spin2',
    'spin3',
    'photoresist_bake',
    'temp_softbake',
    'time_softbake'
]
y = 'is_low_yield'

In [4]:
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report

X_train = df_train[photo_softbake].copy()
X_test  = df_test[photo_softbake].copy()
y_train = df_train[y].copy()
y_test  = df_test[y].copy()

model = LGBMClassifier(
    random_state=42,
    objective='binary',
    class_weight='balanced',
    verbosity=-1,
    learning_rate=0.1,
    max_depth=-1,
    min_child_samples=10,
    n_estimators=300,
    num_leaves=63,
)
model.fit(X_train, y_train)

print(classification_report(y_test, model.predict(X_test), zero_division=0))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4028
           1       0.99      0.97      0.98       579

    accuracy                           0.99      4607
   macro avg       0.99      0.98      0.99      4607
weighted avg       0.99      0.99      0.99      4607



In [5]:
from pathlib import Path

save_path = Path("../model/photo_softbake_lgbm.txt")
save_path.parent.mkdir(parents=True, exist_ok=True)
model.booster_.save_model(str(save_path))
print(f"모델 저장 완료: {save_path.resolve()}")

모델 저장 완료: /Users/yujin/Desktop/Yujin/포빅아/pobiga-bigdata-project/model/photo_softbake_lgbm.txt
